In [1]:
from preprocess import TextProcessor
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
import os

# ==============================
# Step 1: Load 20 Newsgroups dataset
# ==============================
categories = None  # use all 20 categories
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories)

print("Training documents:", len(newsgroups_train.data))
print("Testing documents:", len(newsgroups_test.data))
print("Categories:", newsgroups_train.target_names)

# ==============================
# Step 2: Preprocess documents with TextProcessor
# ==============================
def preprocess_docs(raw_docs):
    processed_docs = []
    for i, text in enumerate(raw_docs):
        try:
            # Save doc temporarily to a file (since your TextProcessor expects a file path)
            temp_path = f"temp_doc_{i}.txt"
            with open(temp_path, "w", encoding="utf-8", errors="ignore") as f:
                f.write(text)

            processor = TextProcessor(temp_path)
            processor.process_all()

            # Join tokens back into a string for TF-IDF
            processed_docs.append(" ".join(processor.final_processed))

            # Remove temporary file
            os.remove(temp_path)
        except Exception as e:
            print(f"Error in doc {i}: {e}")
            processed_docs.append("")  # fallback empty doc
    return processed_docs

print("\nPreprocessing training set...")
train_processed = preprocess_docs(newsgroups_train.data[:500])  # limit for speed (try 500 first)

print("\nPreprocessing testing set...")
test_processed = preprocess_docs(newsgroups_test.data[:200])  # limit for speed (try 200 first)



# ==============================
# Step 3: TF-IDF Vectorization
# ==============================
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(train_processed)
X_test_tfidf  = vectorizer.transform(test_processed)

print("\n=== TF-IDF Representation ===")
print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)
print("Sample features:", vectorizer.get_feature_names_out()[:20])


Training documents: 11314
Testing documents: 7532
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']

Preprocessing training set...

Preprocessing testing set...

=== TF-IDF Representation ===
Training shape: (500, 16710)
Testing shape: (200, 16710)
Sample features: ['00' '000' '0001' '0005895485mcimailcom' '002' '00222'
 '002251w5734117130axeacadiauca' '002251waxeacadiauca' '0038' '0053'
 '0067' '008' '0087' '009' '0091' '00m0bmfqq9' '010' '0101' '0105' '0106']


In [3]:
# Import your manual modules
from vocab_builder import VocabularyBuilder
from manual_nb import NaiveBayesClassifier, build_document_term_matrix
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# ==============================
# Step 3: Manual Vocabulary + Term Frequency
# ==============================
vb = VocabularyBuilder(train_processed)
vb.build()

vocabulary = vb.get_vocabulary()
train_term_freqs = vb.get_term_frequencies()

# Training document-term matrix
X_train_manual = build_document_term_matrix(train_term_freqs, vocabulary)

# Testing document-term matrix
test_term_freqs = {i: Counter(doc.split()) for i, doc in enumerate(test_processed)}
X_test_manual = build_document_term_matrix(test_term_freqs, vocabulary)


# ==============================
# Step 4: Train Manual Naive Bayes
# ==============================
nb_manual = NaiveBayesClassifier()
nb_manual.fit(X_train_manual, newsgroups_train.target[:500], feature_names=vocabulary)

# Predict
y_pred_manual = nb_manual.predict(X_test_manual)


# ==============================
# Step 5: Evaluate
# ==============================
accuracy = accuracy_score(newsgroups_test.target[:200], y_pred_manual)
precision = precision_score(newsgroups_test.target[:200], y_pred_manual, average='weighted', zero_division=0)
recall = recall_score(newsgroups_test.target[:200], y_pred_manual, average='weighted', zero_division=0)
f1 = f1_score(newsgroups_test.target[:200], y_pred_manual, average='weighted', zero_division=0)

# print("\n=== Manual Naive Bayes Evaluation ===")
# print(f"Accuracy: {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1-score: {f1:.4f}")


=== Manual Naive Bayes Evaluation ===
Accuracy: 0.5350
Precision: 0.6045
Recall: 0.5350
F1-score: 0.5078


In [5]:
# ==============================
# Library Imports
# ==============================
from sklearn.neighbors import NearestCentroid
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Prepare targets
y_train = newsgroups_train.target
y_test = newsgroups_test.target[:X_test_tfidf.shape[0]]


# ==============================
# Evaluation Function
# ==============================
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

# ==============================
# Subset for quick testing
# ==============================
X_train_subset = X_train_tfidf[:500]
y_train_subset = y_train[:500]
X_test_subset = X_test_tfidf[:200]
y_test_subset = y_test[:200]

# ==============================
# 1️⃣ Rocchio (Nearest Centroid)
# ==============================
rocchio_model = NearestCentroid(metric='euclidean')
rocchio_model.fit(X_train_subset, y_train_subset)
evaluate_model(rocchio_model, X_test_subset, y_test_subset, "Rocchio")

# ==============================
# 2️⃣ Maximum Entropy (Logistic Regression)
# ==============================
maxent_model = LogisticRegression(solver='lbfgs', max_iter=1000)
maxent_model.fit(X_train_subset, y_train_subset)
evaluate_model(maxent_model, X_test_subset, y_test_subset, "Maximum Entropy")

# ==============================
# 3️⃣ SVM (LinearSVC)
# ==============================
svm_model = LinearSVC(max_iter=10000)
svm_model.fit(X_train_subset, y_train_subset)
evaluate_model(svm_model, X_test_subset, y_test_subset, "SVM")


In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Dictionary to store results
results = {}

def evaluate_model(name, y_true, y_pred):
    results[name] = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "F1-score": f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

# === Evaluate Naive Bayes ===
y_pred_nb = nb_manual.predict(X_test_manual)
evaluate_model("Naive Bayes", y_test, y_pred_nb)

# === Evaluate Rocchio ===
y_pred_rocchio = rocchio_model.predict(X_test_tfidf)
evaluate_model("Rocchio", y_test, y_pred_rocchio)

# === Evaluate Maximum Entropy (Logistic Regression) ===
y_pred_maxent = maxent_model.predict(X_test_tfidf)
evaluate_model("Max Entropy", y_test, y_pred_maxent)

# === Evaluate SVM ===
y_pred_svm = svm_model.predict(X_test_tfidf)
evaluate_model("SVM", y_test, y_pred_svm)

# Convert results to DataFrame
df_results = pd.DataFrame(results).T  # transpose so algorithms are rows

print("\n=== Model Comparison ===")
print(df_results.round(4))



=== Model Comparison ===
             Accuracy  Precision  Recall  F1-score
Naive Bayes     0.535     0.6045   0.535    0.5078
Rocchio         0.555     0.6443   0.555    0.5501
Max Entropy     0.460     0.5996   0.460    0.4299
SVM             0.610     0.6365   0.610    0.5984
